# SQL Complete Practice — Part 09: SQL Set Operations
**Topics 147–151 | MySQL | Ankanhaldar**

This notebook covers combining the results of multiple `SELECT` statements using set operations and understanding how these operations behave in MySQL.

> **Execution note:** These are MySQL SQL cells, not Python. Run them in MySQL Workbench, VS Code with SQLTools, or a compatible SQL notebook/kernel. A standard Python kernel does not execute SQL directly.

> **Practice database:** Run the setup cell first. The sample tables contain student records from two academic batches.

In [ ]:
CREATE DATABASE IF NOT EXISTS ankan_sql_practice;
USE ankan_sql_practice;

CREATE TABLE IF NOT EXISTS set_demo_batch_a (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(80) NOT NULL,
    course VARCHAR(60)
);

CREATE TABLE IF NOT EXISTS set_demo_batch_b (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(80) NOT NULL,
    course VARCHAR(60)
);

INSERT INTO set_demo_batch_a (student_id, student_name, course)
VALUES
    (1, 'Amit', 'MCA'),
    (2, 'Riya', 'MCA'),
    (3, 'Suman', 'BCA'),
    (4, 'Neha', 'MCA')
ON DUPLICATE KEY UPDATE
    student_name = VALUES(student_name),
    course = VALUES(course);

INSERT INTO set_demo_batch_b (student_id, student_name, course)
VALUES
    (3, 'Suman', 'BCA'),
    (4, 'Neha', 'MCA'),
    (5, 'Priya', 'BCA'),
    (6, 'Rahul', 'MCA')
ON DUPLICATE KEY UPDATE
    student_name = VALUES(student_name),
    course = VALUES(course);

SELECT * FROM set_demo_batch_a;
SELECT * FROM set_demo_batch_b;

**Setup note:** Both tables intentionally share student IDs 3 and 4. The examples use matching column counts and compatible data types, as required by set operations.

## Topic 147: What Are SQL Set Operations?

Set operations combine the results of two or more `SELECT` statements into one result.

Important rules:
- Each `SELECT` must return the same number of columns.
- Corresponding columns should have compatible data types.
- Column names in the final result are generally taken from the first `SELECT`.
- `ORDER BY` is applied to the combined result and is normally written at the end.

In MySQL, `UNION` and `UNION ALL` are supported. MySQL 8.0.31 and later also support `INTERSECT` and `EXCEPT`.

### SQL Example — Topic 147

In [ ]:
SELECT student_id, student_name, course
FROM set_demo_batch_a

UNION

SELECT student_id, student_name, course
FROM set_demo_batch_b;

**Expected result / explanation**

The query combines rows from both batches and removes duplicate result rows because it uses `UNION`.

## Topic 148: UNION Operator

`UNION` combines the results of two or more `SELECT` statements and removes duplicate rows from the combined result.

Use `UNION` when you need a distinct combined result. Duplicate removal may require extra processing compared with `UNION ALL`.

### SQL Example — Topic 148

In [ ]:
SELECT student_id, student_name, course
FROM set_demo_batch_a

UNION

SELECT student_id, student_name, course
FROM set_demo_batch_b
ORDER BY student_id;

**Expected result / explanation**

The result contains each distinct row once. Students appearing identically in both tables are not repeated.

## Topic 149: UNION ALL Operator

`UNION ALL` combines the results of multiple `SELECT` statements and keeps all rows, including duplicates.

It is generally faster than `UNION` when duplicate removal is not required because it does not need to eliminate duplicate rows.

### SQL Example — Topic 149

In [ ]:
SELECT student_id, student_name, course
FROM set_demo_batch_a

UNION ALL

SELECT student_id, student_name, course
FROM set_demo_batch_b
ORDER BY student_id;

**Expected result / explanation**

Students 3 and 4 appear twice because their rows are present in both source tables and `UNION ALL` retains duplicates.

## Topic 150: INTERSECT Operator

`INTERSECT` returns only rows that appear in both query results.

MySQL supports `INTERSECT` starting with MySQL 8.0.31. It removes duplicates by default. On older MySQL versions, use an alternative such as an inner join or `EXISTS`.

The corresponding columns must have compatible types and the same count.

### SQL Example — Topic 150

In [ ]:
-- MySQL 8.0.31+
SELECT student_id, student_name, course
FROM set_demo_batch_a

INTERSECT

SELECT student_id, student_name, course
FROM set_demo_batch_b
ORDER BY student_id;

**Expected result / explanation**

Only identical rows found in both batch tables are returned. In this sample, Suman and Neha are common to both.

## Topic 151: EXCEPT Operator

`EXCEPT` returns rows from the first query that do not appear in the second query.

MySQL supports `EXCEPT` starting with MySQL 8.0.31. It removes duplicates by default. On older MySQL versions, use `NOT EXISTS` or a suitable anti-join pattern.

The order of the two `SELECT` statements matters: `A EXCEPT B` is not the same as `B EXCEPT A`.

### SQL Example — Topic 151

In [ ]:
-- MySQL 8.0.31+
SELECT student_id, student_name, course
FROM set_demo_batch_a

EXCEPT

SELECT student_id, student_name, course
FROM set_demo_batch_b
ORDER BY student_id;

**Expected result / explanation**

The result contains rows in batch A that are not present in batch B. In this sample, Amit and Riya are returned.

## MySQL Alternatives for Older Versions

If you are using a MySQL version older than 8.0.31, use these alternatives for `INTERSECT` and `EXCEPT`.

### INTERSECT alternative using INNER JOIN

```sql
SELECT a.student_id, a.student_name, a.course
FROM set_demo_batch_a AS a
INNER JOIN set_demo_batch_b AS b
    ON a.student_id = b.student_id
   AND a.student_name = b.student_name
   AND a.course = b.course;
```

### EXCEPT alternative using NOT EXISTS

```sql
SELECT a.student_id, a.student_name, a.course
FROM set_demo_batch_a AS a
WHERE NOT EXISTS (
    SELECT 1
    FROM set_demo_batch_b AS b
    WHERE b.student_id = a.student_id
      AND b.student_name = a.student_name
      AND b.course = a.course
);
```

These examples compare all three columns, matching the full-row behaviour of the set-operation examples.

## Set Operations vs JOIN

| Set Operations | JOIN |
|---|---|
| Combine rows vertically | Combine columns horizontally |
| Queries need matching column counts | Tables need a join condition (except CROSS JOIN) |
| Examples: UNION, INTERSECT, EXCEPT | Examples: INNER, LEFT, RIGHT JOIN |
| Results are stacked | Related columns are placed side by side |

## Quick Revision Table

| Operator | Result |
|---|---|
| `UNION` | Combines results and removes duplicates |
| `UNION ALL` | Combines results and keeps duplicates |
| `INTERSECT` | Returns rows common to both results |
| `EXCEPT` | Returns rows in the first result but not the second |

## Important Interview Points

1. Both sides of a set operation must return the same number of columns.
2. Corresponding columns must have compatible data types.
3. `UNION` removes duplicate rows; `UNION ALL` does not.
4. MySQL supports `INTERSECT` and `EXCEPT` from version 8.0.31.
5. `EXCEPT` is order-sensitive.
6. `ORDER BY` should be applied at the end of the combined query.
7. Set operations combine rows, while joins combine columns.

## Practice Checklist

- [ ] Combine two result sets using UNION.
- [ ] Compare UNION with UNION ALL.
- [ ] Find common rows using INTERSECT.
- [ ] Find rows present only in the first result using EXCEPT.
- [ ] Write an INNER JOIN alternative for INTERSECT.
- [ ] Write a NOT EXISTS alternative for EXCEPT.
- [ ] Explain set operations vs JOIN.

**End of Part 09 — Topics 147–151.**